**Laboratorio #2 - Representaciones básicas de texto**

Juan Diego Letona Aguilar

*20230285*

In [3]:
# Parcialmente reciclado del notebook 1
# la primera parte (imports, carga de paquetes, etc. es practicamente igual a la iteracion anterior)

#principalmente librerias y carga de paquetes necesarios para el proyecto

import sys
import subprocess

packages = [
    "pandas",
    "numpy",
    "nltk",
    "matplotlib",
    "wordcloud",
    "spacy",
    "tqdm"
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U", *packages
])

# Modelo pequeño de spaCy para español
subprocess.check_call([
    sys.executable, "-m", "spacy", "download", "es_core_news_sm"
])

0

In [4]:
from pathlib import Path
from collections import Counter
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

#voy a usar spacy porque el corpus es en español y spacy tiene un modelo para español
#otros modelos suelen funcionar mejor para el idioma ingles
#depende del caso de uso

import spacy
from tqdm.auto import tqdm
from wordcloud import WordCloud

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

#recursos de NLTK
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [5]:
import pandas as pd

df = pd.read_csv("df_total.csv")
#leemos el archivo csv y lo guardamos en un DataFrame de pandas

print("Dimensiones del DataFrame:", df.shape)
display(df.head())

Dimensiones del DataFrame: (1217, 3)


,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresarial para el desarrollo sostenible el director de sostenibilidad y clientes globales de BBVA en Colomb...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domingo que buscará una cooperación más estrecha con su par estadounidense y que apoyará las salidas a bo...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como lo es la aviación Viva presentó su avión rosado A320NEO que apuesta por la equidad de género la luc...,Alianzas
3,https://www.larepublica.co/redirect/post/3342889,Con el dato de marzo el IPC interanual encadena su decimoquinta tasa positiva consecutiva. La inflación publicada por el INE se ha mantenido igual...,Macroeconomia
4,https://www.larepublica.co/redirect/post/3427208,Ayer en Cartagena se dio inicio a la versión número 56 de la Convención Bancaria. Este será el primer encuentro de los banqueros del país con el p...,Otra


In [6]:
#configuracion y descripcion de columnas

URL_COL = "url"
TEXT_COL = "news"
CATEGORY_COL = "Type"

descripcion = {
    URL_COL: "enlace original de la noticia",
    TEXT_COL: "contenido textual de la noticia",
    CATEGORY_COL: "categoria asignada a la noticia"
}

columnas = pd.DataFrame({
    "columna": descripcion.keys(),
    "tipo_de_dato": [str(df[col].dtype) for col in descripcion],
    "informacion": descripcion.values()
})

display(
    columnas.style
    .hide(axis="index")
    .set_caption("Columnas del corpus")
)

columna,tipo_de_dato,informacion
url,str,enlace original de la noticia
news,str,contenido textual de la noticia
Type,str,categoria asignada a la noticia


In [7]:
#preparamos el corpus para el procesamiento

corpus = df[[URL_COL, TEXT_COL, CATEGORY_COL]].copy()

corpus[TEXT_COL] = corpus[TEXT_COL].fillna("").astype(str)
corpus = corpus[corpus[TEXT_COL].str.strip().ne("")]
corpus = corpus.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

stopwords_es = set(stopwords.words("spanish"))

#basicamente quitamos la puntuacion de los tokens, para que no afecte el conteo de palabras y tipos de palabras

def quitar_puntuacion(token):
    return "".join(
        caracter for caracter in token
        if not unicodedata.category(caracter).startswith("P")
    ).strip()

def contar_tokens_tipos(columna):
    tokens = [token for documento in columna for token in documento]
    return len(tokens), len(set(tokens))

print("Documentos que serán procesados:", len(corpus))

Documentos que serán procesados: 1137


In [8]:
#tokenizacion

tqdm.pandas(desc="Tokenizando")

corpus["tokens_tokenizados"] = corpus[TEXT_COL].progress_apply(
    lambda texto: word_tokenize(texto, language="spanish")
)

#minusculas

corpus["tokens_minusculas"] = corpus["tokens_tokenizados"].apply(
    lambda tokens: [token.lower() for token in tokens]
)

#eliminacion de puntuacion

corpus["tokens_sin_puntuacion"] = corpus["tokens_minusculas"].apply(
    lambda tokens: [
        limpio for token in tokens
        if (limpio := quitar_puntuacion(token))
    ]
)

#eliminacion de stopwords

corpus["tokens_sin_stopwords"] = corpus["tokens_sin_puntuacion"].apply(
    lambda tokens: [token for token in tokens if token not in stopwords_es]
)

Tokenizando: 100%|██████████| 1137/1137 [00:05<00:00, 213.65it/s]


In [9]:
#lematizacion
nlp = spacy.load(
    "es_core_news_sm",
    disable=["parser", "ner"]
)
textos = corpus["tokens_sin_stopwords"].apply(" ".join)
corpus["tokens_lematizados"] = [
    [
        token.lemma_.lower()
        for token in documento
        if token.lemma_.strip()
    ]
    for documento in tqdm(
        nlp.pipe(textos, batch_size=64),
        total=len(textos),
        desc="Lematizando"
    )
]
#tokens y tipos despues de cada etapa
etapas = {
    "Tokenización": "tokens_tokenizados",
    "Minúsculas": "tokens_minusculas",
    "Sin puntuación": "tokens_sin_puntuacion",
    "Sin stopwords": "tokens_sin_stopwords",
    "Lematización": "tokens_lematizados"
}
resultados = []
for etapa, columna in etapas.items():
    tokens, tipos = contar_tokens_tipos(corpus[columna])

    resultados.append({
        "etapa": etapa,
        "tokens": tokens,
        "tipos": tipos
    })

resumen_etapas = pd.DataFrame(resultados)
resumen_etapas["cambio_tokens"] = (
    resumen_etapas["tokens"].diff()
)
resumen_etapas["reduccion_vocabulario_%"] = (
    resumen_etapas["tipos"]
    .pct_change()
    .mul(-100)
    .round(2)
)
display(
    resumen_etapas.style
    .hide(axis="index")
    .set_caption("Tokens y tipos después de cada etapa")
)
#cambio producido durante la lematizacion
tokens_antes_lematizar = resumen_etapas.loc[
    resumen_etapas["etapa"] == "Sin stopwords",
    "tokens"
].iloc[0]

tokens_despues_lematizar = resumen_etapas.loc[
    resumen_etapas["etapa"] == "Lematización",
    "tokens"
].iloc[0]
diferencia_lematizacion = (
    tokens_despues_lematizar - tokens_antes_lematizar
)
if diferencia_lematizacion > 0:
    print(
        f"La lematización produjo un aumento de "
        f"{diferencia_lematizacion:,} tokens."
    )
elif diferencia_lematizacion < 0:
    print(
        f"La lematización produjo una reducción de "
        f"{abs(diferencia_lematizacion):,} tokens."
    )
else:
    print("La lematización no cambió la cantidad de tokens.")

Lematizando: 100%|██████████| 1137/1137 [00:44<00:00, 25.74it/s]


etapa,tokens,tipos,cambio_tokens,reduccion_vocabulario_%
Tokenización,631740,38941,nan,nan
Minúsculas,631740,36093,0.000000,7.310000
Sin puntuación,598600,35460,-33140.000000,1.750000
Sin stopwords,319034,35248,-279566.000000,0.600000
Lematización,319318,28292,284.000000,19.730000


La lematización produjo un aumento de 284 tokens.
